# 👗 Women's Clothing Customer Analytics

This notebook analyzes women's clothing e-commerce review data and builds a simple recommendation classifier.

The workflow is based on the original capstone notebook, cleaned up for a portable GitHub project.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

DATA_PATH = Path("data/Womens Clothing E-Commerce Reviews.csv")


## 1. Load Data


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Add 'Womens Clothing E-Commerce Reviews.csv' inside the data/ folder."
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())


## 2. Data Understanding


In [ ]:
df.info()


In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
display(missing.to_frame("missing_values"))


## 3. Data Cleaning


In [ ]:
clean = df.copy()

if "Unnamed: 0" in clean.columns:
    clean = clean.drop(columns=["Unnamed: 0"])

clean["Title"] = clean["Title"].fillna("No Title")
clean["Review Text"] = clean["Review Text"].fillna("No Review")

# Remove rows still missing business/category fields.
clean = clean.dropna().reset_index(drop=True)

print("Cleaned rows:", len(clean))
display(clean.head())


## 4. Customer Rating Analysis


In [ ]:
print("Average age:", round(clean["Age"].mean(), 2))
print("Average rating:", round(clean["Rating"].mean(), 2))
print("Recommendation rate:", round(clean["Recommended IND"].mean() * 100, 2), "%")
print("Total reviews:", len(clean))


In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x="Rating", data=clean)
plt.title("Customer Ratings")
plt.xlabel("Rating")
plt.ylabel("Review Count")
plt.tight_layout()
plt.show()


## 5. Customer Age Analysis


In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(clean["Age"], bins=20)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.tight_layout()
plt.show()


## 6. Department Performance


In [ ]:
department_reviews = clean["Department Name"].value_counts()
display(department_reviews.to_frame("review_count"))

plt.figure(figsize=(8,5))
department_reviews.plot(kind="bar")
plt.title("Reviews by Department")
plt.ylabel("Review Count")
plt.tight_layout()
plt.show()


In [ ]:
dept_rating = (
    clean.groupby("Department Name")["Rating"]
         .mean()
         .sort_values(ascending=False)
)

display(dept_rating.to_frame("average_rating"))

plt.figure(figsize=(8,5))
dept_rating.plot(kind="bar")
plt.title("Average Rating by Department")
plt.ylabel("Average Rating")
plt.tight_layout()
plt.show()


## 7. Recommendation Analysis


In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(x="Recommended IND", data=clean)
plt.title("Recommendation Distribution")
plt.xlabel("Recommended")
plt.ylabel("Review Count")
plt.tight_layout()
plt.show()


## 8. Top Clothing Classes


In [ ]:
top_classes = clean["Class Name"].value_counts().head(10)
display(top_classes.to_frame("review_count"))

plt.figure(figsize=(9,5))
top_classes.plot(kind="bar")
plt.title("Top 10 Clothing Classes")
plt.ylabel("Review Count")
plt.tight_layout()
plt.show()


## 9. Recommendation Classification

The original capstone used:
- Age
- Rating
- Positive Feedback Count

to predict whether a product was recommended.


In [ ]:
X = clean[["Age", "Rating", "Positive Feedback Count"]]
y = clean["Recommended IND"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("\nClassification Report:\n")
print(classification_report(y_test, pred))


In [ ]:
cm = confusion_matrix(y_test, pred)

plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title("Decision Tree Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i,j], ha="center", va="center")

plt.tight_layout()
plt.show()


## 10. Key Findings from the Original Capstone Run

The uploaded notebook recorded:

- **23,472** cleaned reviews
- Average rating of approximately **4.20**
- Recommendation rate of approximately **82.23%**
- `Tops` had the largest number of reviews
- `Dresses` and `Knits` were among the most-reviewed clothing classes
- The original Decision Tree run recorded approximately **93.06% test accuracy**

These values should be regenerated when running this cleaned notebook on the same source dataset.
